<a href="https://colab.research.google.com/github/Mahian-Hoq/ML_Journey/blob/RAG_Implementations/From_document_to_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

I will use my thesis paper, (chapters only) as my resource file. From where I will be retriving data using RAG, which will be compatibe with my query.

In [ ]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 32.0 MB/s eta 0:00:00


In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 57.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

drive.mount('/content/MyDrive/')

Mounted at /content/MyDrive/


In [ ]:
import pymupdf
pdf_path = '/content/MyDrive/MyDrive/Varsity/Thesis/Only_Chapters.pdf'
doc = pymupdf.open(pdf_path)

print('Number of Pages: ', len(doc))

Number of Pages:  73


In [ ]:
all_text = ""

for page in doc:
  all_text += page.get_text()
  all_text += "\n"

print(all_text[:3000])

CHAPTER 1 
INTRODUCTION 
1.1 Background and Motivation 
Public Place Cleanliness is an important aspect of Environmental Sustainability, public health 
and city management [1], [6], [13]. “Litter” and “waste disposal” is experienced by many at 
public places such as parks, schools, transportation hubs, residential neighborhoods and 
roadsides. If left uncontrolled, the waste may have a negative effect on the environment, health, 
aesthetics, and overall ecological issues. Therefore, up to the disposal of the municipalities, 
institutions and environmental management authorities the monitoring and cleaning of public 
spaces has become an indispensable task. 
The majority of cleanliness monitoring relies on traditional way of human observation and 
manual inspections [6], [7]. Consider an assignment to areas and the collection of waste 
materials may be needed as well as assessing environmental conditions. These can be used on 
small scale, but in larger scale, it gets trickier. Manual m

In [ ]:
chunk_size = 1000

chunks = []

for i in range(0,len(all_text), chunk_size):
  chunk = all_text[i:i + chunk_size]
  chunks.append(chunk)

print("Number of chunks: ", len(chunks))

Number of chunks:  134


In [ ]:
for i, chunk in enumerate(chunks[:5]):
  print(f"\n----CHUNK {i}------")
  print(chunk)


----CHUNK 0------
CHAPTER 1 
INTRODUCTION 
1.1 Background and Motivation 
Public Place Cleanliness is an important aspect of Environmental Sustainability, public health 
and city management [1], [6], [13]. “Litter” and “waste disposal” is experienced by many at 
public places such as parks, schools, transportation hubs, residential neighborhoods and 
roadsides. If left uncontrolled, the waste may have a negative effect on the environment, health, 
aesthetics, and overall ecological issues. Therefore, up to the disposal of the municipalities, 
institutions and environmental management authorities the monitoring and cleaning of public 
spaces has become an indispensable task. 
The majority of cleanliness monitoring relies on traditional way of human observation and 
manual inspections [6], [7]. Consider an assignment to areas and the collection of waste 
materials may be needed as well as assessing environmental conditions. These can be used on 
small scale, but in larger scale, it gets

Here we can see the chunks are separated but some times the words are separated too. This will hurt our intention to vectorize them.

###Block Structure

In [ ]:
#clearing chunks
chunks = []

In [ ]:
for page in doc:
  blocks = page.get_text('blocks')

  for block in blocks:
    text = block[4]
    chunks.append(text)

for i, chunk in enumerate(chunks[:5]):
  print(f"\n----CHUNK {i}------")
  print(chunk)


----CHUNK 0------
CHAPTER 1 


----CHUNK 1------
INTRODUCTION 


----CHUNK 2------
1.1 Background and Motivation 


----CHUNK 3------
Public Place Cleanliness is an important aspect of Environmental Sustainability, public health 
and city management [1], [6], [13]. “Litter” and “waste disposal” is experienced by many at 
public places such as parks, schools, transportation hubs, residential neighborhoods and 
roadsides. If left uncontrolled, the waste may have a negative effect on the environment, health, 
aesthetics, and overall ecological issues. Therefore, up to the disposal of the municipalities, 
institutions and environmental management authorities the monitoring and cleaning of public 
spaces has become an indispensable task. 


----CHUNK 4------
The majority of cleanliness monitoring relies on traditional way of human observation and 
manual inspections [6], [7]. Consider an assignment to areas and the collection of waste 
materials may be needed as well as assessing environme

Here we can see the Heading are being separated as chunks here. So we can implement a logic that will say

> If a block is very short, treat it as a possible heading/caption and attach it to the following block.

In [ ]:
#filtering function

def is_valid_block(text):

    text = text.strip()

    # Empty block
    if not text:
        return False

    # Page number
    if text.isdigit():
        return False

    # Daffodil footer
    if "©DAFFODIL INTERNATIONAL UNIVERSITY" in text:
        return False

    # Figure caption
    if text.lower().startswith("figure"):
        return False

    return True

In [ ]:
chunks = []

for page in doc:

    blocks = page.get_text("blocks")

    i = 0

    # While loop is used because sometimes we jump 2 blocks,
    # sometimes 1 block
    while i < len(blocks):

        text = blocks[i][4].strip()

        # FILTER USELESS BLOCKS
        if not is_valid_block(text):
            i += 1
            continue

        # CHUNKING LOGIC
        # If block is short, merge it with next block
        if len(text) < 50 and i + 1 < len(blocks):

            next_text = blocks[i + 1][4].strip()

            combined_text = text + "\n" + next_text

            chunks.append(combined_text)

            i += 2

        else:

            chunks.append(text)

            i += 1

In [ ]:
for i, chunk in enumerate(chunks[:50]):
  print(f"\n----CHUNK {i}------")
  print(chunk)


----CHUNK 0------
CHAPTER 1
INTRODUCTION

----CHUNK 1------
1.1 Background and Motivation
Public Place Cleanliness is an important aspect of Environmental Sustainability, public health 
and city management [1], [6], [13]. “Litter” and “waste disposal” is experienced by many at 
public places such as parks, schools, transportation hubs, residential neighborhoods and 
roadsides. If left uncontrolled, the waste may have a negative effect on the environment, health, 
aesthetics, and overall ecological issues. Therefore, up to the disposal of the municipalities, 
institutions and environmental management authorities the monitoring and cleaning of public 
spaces has become an indispensable task.

----CHUNK 2------
The majority of cleanliness monitoring relies on traditional way of human observation and 
manual inspections [6], [7]. Consider an assignment to areas and the collection of waste 
materials may be needed as well as assessing environmental conditions. These can be used on 
small s

###Embedding

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
chunk_embed = embed_model.encode(chunks)

print('Number of Chunks: ',len(chunks))
print('Embedding shape: ', chunk_embed.shape)
print('First vector dimentions', len(chunk_embed[0]))

Number of Chunks:  443
Embedding shape:  (443, 384)
First vector dimentions 384


###Now storing it into the FAISS indexs

In [ ]:
import faiss
import numpy as np

In [ ]:
chunk_embed = np.array(chunk_embed).astype('float32')

In [ ]:
index = faiss.IndexFlatL2(chunk_embed.shape[1])

In [ ]:
index.add(chunk_embed)

index.ntotal

443

###Query

In [ ]:
query = "How is the waste coverage calculated"

In [ ]:
query_embed = np.array(embed_model.encode([query])).astype('float32')

query_embed.shape

(1, 384)

###Searching

In [ ]:
k = 3

distance, indices = index.search(query_embed, k)

print(f'Distances: {distance}, Indices: {indices}')

Distances: [[0.27559072 0.31274518 0.538573  ]], Indices: [[ 36 262 269]]


###Retrival

In [ ]:
retrived_chunks = []

for idx in indices[0]:
  retrived_chunks.append(chunks[idx])

In [ ]:
for i, chunk in enumerate(retrived_chunks):
  print(f'\n----- RESULT {i+1} -----')
  print(chunk)


----- RESULT 1 -----
A Waste Coverage Estimation Method was introduced in:

----- RESULT 2 -----
The waste coverage ratio is calculated using Equation (3.1).

----- RESULT 3 -----
Waste Coverage (%) 
Cleanliness Category
0.00 – 0.75 
Clean


> RAG retrieval works, but retrieval quality depends heavily on chunking quality.